# Práctica 2 · Regresión logística y métricas

Objetivo: diferenciar probabilidad estimada, umbral y errores. Primera parte: ejemplo binario sintético. Segunda: Wine multiclase con las particiones compartidas.

**Fuentes:** Géron, 3.ª ed., caps. 2–5; Grus, 2.ª ed., caps. 13–16; Gallatin y Albon, Cookbook adjunto, caps. 3–6 (preprocesamiento). Las consignas y los datos sintéticos son elaboración didáctica. Véase `../apuntes.md` y `../datos/README.md`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
ROOT = Path.cwd()
if not (ROOT / 'datos').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'datos').is_dir():
    raise FileNotFoundError('Abra el notebook desde la carpeta del subtema o desde notebooks, conservando datos/.')
print('Versiones:', sklearn.__version__, np.__version__, pd.__version__)

Versiones: 1.8.0 2.3.5 2.2.3


## 1. Probabilidad y umbral
Datos inventados exclusivamente para calcular métricas. La clase positiva vale 1.

In [2]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
p=np.array([.1,.4,.55,.8,.9,.7]);y=np.array([0,0,1,1,0,1])
for t in [.5,.75]:
    pred=(p>=t).astype(int)
    print('Umbral:',t,'Predicciones:',pred)
    print(confusion_matrix(y,pred,labels=[0,1]))
    print('Exactitud, precisión, sensibilidad, F1:',accuracy_score(y,pred),precision_score(y,pred,zero_division=0),recall_score(y,pred),f1_score(y,pred))

Umbral: 0.5 Predicciones: [0 0 1 1 1 1]
[[2 1]
 [0 3]]
Exactitud, precisión, sensibilidad, F1: 0.8333333333333334 0.75 1.0 0.8571428571428571
Umbral: 0.75 Predicciones: [0 0 0 1 1 0]
[[2 1]
 [2 1]]
Exactitud, precisión, sensibilidad, F1: 0.5 0.5 0.3333333333333333 0.4


**Ejercicio P2-A:** explique qué errores cambiaron y cuál umbral preferiría si omitir un positivo tiene costo alto. La elección necesita costos y validación, no solo intuición.

## 2. Wine: clasificación multiclase
Tres cultivares, no diagnóstico médico. Se utiliza F1 macro para dar el mismo peso a cada clase.

In [3]:
df = pd.read_csv(ROOT / 'datos/wine.csv')
features = [c for c in df.columns if c not in ['clase', 'particion']]
train = df[df.particion == 'entrenamiento']
val = df[df.particion == 'validacion']
Xtr, ytr = train[features], train.clase
Xv, yv = val[features], val.clase
print('Entrenamiento:', Xtr.shape, 'Validación:', Xv.shape)

Entrenamiento: (106, 13) Validación: (36, 13)


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
model=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=3000,random_state=43))
model.fit(Xtr,ytr);pred=model.predict(Xv)
print(classification_report(yv,pred,zero_division=0,digits=4))
print('Matriz: filas reales, columnas predichas')
print(confusion_matrix(yv,pred,labels=[0,1,2]))
print('Probabilidades de los primeros tres ejemplos:\n',model.predict_proba(Xv.iloc[:3]))

              precision    recall  f1-score   support

           0     1.0000    0.9167    0.9565        12
           1     0.9333    1.0000    0.9655        14
           2     1.0000    1.0000    1.0000        10

    accuracy                         0.9722        36
   macro avg     0.9778    0.9722    0.9740        36
weighted avg     0.9741    0.9722    0.9721        36

Matriz: filas reales, columnas predichas
[[11  1  0]
 [ 0 14  0]
 [ 0  0 10]]
Probabilidades de los primeros tres ejemplos:
 [[9.92303471e-01 7.48563399e-03 2.10895202e-04]
 [9.97179270e-01 2.27269191e-03 5.48038166e-04]
 [9.99738214e-01 2.53208173e-04 8.57811193e-06]]


**Entregable:** indique una confusión concreta entre clases, interprete F1 macro y explique por qué no se aplica un umbral binario único sin reformular el problema.

**Extensión P2-B:** cambie C entre 0.1, 1 y 10, compare solo validación y registre el efecto. Se desarrollará selección sistemática en 4.5.